# UC02 — Detección de Accidentes Simulados

## Objetivos de Aprendizaje

1. **Modelar redes de relaciones** entre siniestros, talleres y proveedores médicos
2. **Generar datos sintéticos de tramas** de fraude coordinado
3. **Construir variables de red** (frecuencia, coincidencias, patrones)
4. **Entrenar clasificador** de tramas con ML.CLASSIFICATION
5. **Visualizar redes sospechosas** en Streamlit

---

## Qué Construirás

Un **detector de tramas de fraude coordinado** que identifica accidentes simulados analizando
conexiones entre siniestros, talleres y proveedores médicos.

**Valor de negocio**: Desmantelar redes de fraude organizado que representan el 40% de las pérdidas por fraude.

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS ACCIDENTES_SIMULADOS_DB;
CREATE SCHEMA IF NOT EXISTS ACCIDENTES_SIMULADOS_DB.PUBLIC;
USE SCHEMA ACCIDENTES_SIMULADOS_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH
    WITH WAREHOUSE_SIZE = 'SMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() AS base_datos, CURRENT_SCHEMA() AS esquema, CURRENT_WAREHOUSE() AS warehouse;

---

## Paso 2: Generar Red de Talleres y Proveedores Médicos

Creamos talleres (50) y proveedores médicos (30) con indicadores de sospecha.

In [ ]:
CREATE OR REPLACE TABLE talleres AS
SELECT
    'TAL' || LPAD(SEQ4(), 3, '0') AS taller_id,
    'Taller_' || SEQ4() AS nombre,
    CASE WHEN UNIFORM(1,100,RANDOM()) <= 20 THEN 1 ELSE 0 END AS es_sospechoso,
    CASE WHEN es_sospechoso = 1 THEN UNIFORM(15,40,RANDOM()) ELSE UNIFORM(1,8,RANDOM()) END AS num_siniestros_mes,
    CASE MOD(SEQ4(), 5)
        WHEN 0 THEN 'Madrid' WHEN 1 THEN 'Barcelona' WHEN 2 THEN 'Valencia'
        WHEN 3 THEN 'Sevilla' ELSE 'Bilbao'
    END AS provincia
FROM TABLE(GENERATOR(ROWCOUNT => 50));

CREATE OR REPLACE TABLE proveedores_medicos AS
SELECT
    'MED' || LPAD(SEQ4(), 3, '0') AS proveedor_id,
    'Clinica_' || SEQ4() AS nombre,
    CASE WHEN UNIFORM(1,100,RANDOM()) <= 20 THEN 1 ELSE 0 END AS es_sospechoso,
    CASE WHEN es_sospechoso = 1 THEN UNIFORM(20,60,RANDOM()) ELSE UNIFORM(2,12,RANDOM()) END AS facturas_mes
FROM TABLE(GENERATOR(ROWCOUNT => 30));

SELECT * FROM talleres WHERE es_sospechoso = 1;

---

## Paso 3: Generar Siniestros Vinculados

2.000 siniestros con relaciones entre tomadores, talleres y proveedores médicos.
- **Tramas (25%)**: Mismo taller sospechoso, coincidencia de testigos, proveedores recurrentes
- **Legítimos (75%)**: Patrones normales sin conexiones sospechosas

In [ ]:
CREATE OR REPLACE TABLE tomadores_red AS
SELECT
    'TOM' || LPAD(SEQ4(), 5, '0') AS tomador_id,
    UNIFORM(22, 70, RANDOM()) AS edad,
    CASE WHEN UNIFORM(0,1,RANDOM()) < 0.5 THEN 'M' ELSE 'F' END AS genero
FROM TABLE(GENERATOR(ROWCOUNT => 800));

CREATE OR REPLACE TABLE siniestros_red AS
SELECT
    'SIN' || LPAD(SEQ4(), 6, '0') AS siniestro_id,
    t.tomador_id,
    CASE WHEN UNIFORM(0,1,RANDOM()) < 0.25 THEN 1 ELSE 0 END AS es_trama,
    DATEADD(day, -UNIFORM(1, 365, RANDOM()), CURRENT_DATE()) AS fecha,
    CASE WHEN es_trama = 1
        THEN (SELECT taller_id FROM talleres WHERE es_sospechoso=1 ORDER BY RANDOM() LIMIT 1)
        ELSE (SELECT taller_id FROM talleres ORDER BY RANDOM() LIMIT 1)
    END AS taller_id,
    CASE WHEN es_trama = 1
        THEN (SELECT proveedor_id FROM proveedores_medicos WHERE es_sospechoso=1 ORDER BY RANDOM() LIMIT 1)
        ELSE (SELECT proveedor_id FROM proveedores_medicos ORDER BY RANDOM() LIMIT 1)
    END AS proveedor_medico_id,
    CASE WHEN es_trama=1 THEN UNIFORM(5000,30000,RANDOM()) ELSE UNIFORM(200,10000,RANDOM()) END::DECIMAL(10,2) AS importe,
    CASE WHEN es_trama=1 THEN UNIFORM(2,4,RANDOM()) ELSE UNIFORM(1,3,RANDOM()) END AS num_vehiculos,
    CASE WHEN es_trama=1 AND UNIFORM(0,1,RANDOM())<0.7 THEN 1 ELSE 0 END AS coincidencia_testigos,
    CASE WHEN es_trama=1 AND UNIFORM(0,1,RANDOM())<0.6 THEN 1 ELSE 0 END AS mismo_taller_repetido
FROM tomadores_red t CROSS JOIN TABLE(GENERATOR(ROWCOUNT => 3))
LIMIT 2000;

SELECT * FROM siniestros_red LIMIT 10;

---

## Paso 4: Ingeniería de Variables de Red

Variables que capturan patrones de conexión entre entidades.

In [ ]:
CREATE OR REPLACE TABLE features_red AS
SELECT
    s.siniestro_id,
    s.importe::FLOAT AS importe,
    s.num_vehiculos::FLOAT AS num_vehiculos,
    s.coincidencia_testigos::FLOAT AS coincidencia_testigos,
    s.mismo_taller_repetido::FLOAT AS mismo_taller_repetido,
    t.num_siniestros_mes::FLOAT AS siniestros_taller_mes,
    t.es_sospechoso::FLOAT AS taller_sospechoso,
    m.facturas_mes::FLOAT AS facturas_proveedor_mes,
    m.es_sospechoso::FLOAT AS proveedor_sospechoso,
    COUNT(*) OVER (PARTITION BY s.tomador_id)::FLOAT AS siniestros_mismo_tomador,
    COUNT(*) OVER (PARTITION BY s.taller_id, DATE_TRUNC('month', s.fecha))::FLOAT AS siniestros_taller_30d,
    s.es_trama
FROM siniestros_red s
JOIN talleres t ON s.taller_id = t.taller_id
JOIN proveedores_medicos m ON s.proveedor_medico_id = m.proveedor_id;

SELECT * FROM features_red LIMIT 10;

---

## Paso 5: Dividir Train/Test

In [ ]:
CREATE OR REPLACE TABLE entrenamiento AS SELECT * FROM features_red SAMPLE (80);
CREATE OR REPLACE TABLE test AS SELECT * FROM features_red WHERE siniestro_id NOT IN (SELECT siniestro_id FROM entrenamiento);

SELECT 'Entrenamiento' AS conjunto, COUNT(*) AS total, SUM(es_trama) AS tramas, ROUND(SUM(es_trama)/COUNT(*)*100,1)||'%' AS tasa FROM entrenamiento
UNION ALL
SELECT 'Test', COUNT(*), SUM(es_trama), ROUND(SUM(es_trama)/COUNT(*)*100,1)||'%' FROM test;

---

## Paso 6: Entrenar Modelo de Detección de Tramas

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION detector_tramas(
    INPUT_DATA     => SYSTEM$REFERENCE('TABLE', 'entrenamiento'),
    TARGET_COLNAME => 'es_trama',
    CONFIG_OBJECT  => {'evaluate': TRUE}
);

---

## Paso 7: Evaluar Rendimiento

In [ ]:
CALL detector_tramas!SHOW_EVALUATION_METRICS();

In [ ]:
CALL detector_tramas!SHOW_CONFUSION_MATRIX();

In [ ]:
CALL detector_tramas!SHOW_FEATURE_IMPORTANCE();

---

## Paso 8: Puntuar Siniestros

- **Trama Probable** (≥70%): Derivar a unidad de fraude organizado
- **Sospechoso** (30-70%): Revisión cruzada
- **Normal** (<30%): Sin acción

In [ ]:
CREATE OR REPLACE TABLE predicciones_tramas AS
SELECT
    siniestro_id,
    detector_tramas!PREDICT(OBJECT_CONSTRUCT(
        'importe', importe, 'num_vehiculos', num_vehiculos,
        'coincidencia_testigos', coincidencia_testigos,
        'mismo_taller_repetido', mismo_taller_repetido,
        'siniestros_taller_mes', siniestros_taller_mes,
        'taller_sospechoso', taller_sospechoso,
        'facturas_proveedor_mes', facturas_proveedor_mes,
        'proveedor_sospechoso', proveedor_sospechoso,
        'siniestros_mismo_tomador', siniestros_mismo_tomador,
        'siniestros_taller_30d', siniestros_taller_30d
    )) AS prediccion,
    prediccion['class']::INT AS trama_predicha,
    prediccion['probability']['1']::FLOAT AS prob_trama,
    CASE
        WHEN prediccion['probability']['1']::FLOAT >= 0.70 THEN 'Trama Probable'
        WHEN prediccion['probability']['1']::FLOAT >= 0.30 THEN 'Sospechoso'
        ELSE 'Normal'
    END AS nivel,
    es_trama AS trama_real
FROM test;

SELECT siniestro_id, ROUND(prob_trama,3) AS prob, nivel, trama_real
FROM predicciones_tramas ORDER BY prob_trama DESC LIMIT 20;

---

## Paso 9: Dashboard Interactivo

In [ ]:
import pandas as pd

session = get_active_session()
st.title('Detección de Accidentes Simulados')
st.markdown('### Análisis de Redes de Fraude — Snowflake Cortex')

df = session.table('predicciones_tramas').to_pandas()

c1,c2,c3,c4 = st.columns(4)
with c1: st.metric('Total Siniestros', len(df))
with c2: st.metric('Tramas Detectadas', len(df[df['NIVEL']=='Trama Probable']))
with c3: st.metric('Exactitud', f"{(df['TRAMA_PREDICHA']==df['TRAMA_REAL']).mean():.1%}")
with c4: st.metric('Tasa Trama Real', f"{df['TRAMA_REAL'].mean():.1%}")

st.markdown('---')
st.subheader('Distribución por Nivel')
rc = df['NIVEL'].value_counts()
st.bar_chart(pd.DataFrame({'Siniestros': rc.values}, index=rc.index))

st.markdown('---')
st.subheader('Siniestros Sospechosos')
filtro = st.multiselect('Filtrar', ['Trama Probable','Sospechoso','Normal'], default=['Trama Probable','Sospechoso'])
fdf = df[df['NIVEL'].isin(filtro)].sort_values('PROB_TRAMA', ascending=False)
fdf['Prob %'] = fdf['PROB_TRAMA'].apply(lambda x: f'{x:.1%}')
st.dataframe(fdf[['SINIESTRO_ID','Prob %','NIVEL','TRAMA_PREDICHA','TRAMA_REAL']], use_container_width=True, height=400)

st.caption('Desarrollado con Snowflake Cortex ML + Streamlit | Datos: Sintéticos')

---

## Paso 10: Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS ACCIDENTES_SIMULADOS_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;